In [ ]:
import json
from openai import OpenAI
import asyncio
import time
import os
import json
import pickle
import base64
from io import BytesIO
from PIL import Image
import numpy as np
from openai import AsyncOpenAI, APIConnectionError, InternalServerError
from asyncio import as_completed
from tqdm import tqdm

import logging
from datetime import datetime, timezone, timedelta


public_deepseek_api_key = os.getenv("PUBLIC_DEEPSEEK_API_KEY")
public_deepseek_base_url = os.getenv("PUBLIC_DEEPSEEK_BASE_URL")
public_deepseek_client = OpenAI(api_key=public_deepseek_api_key,
                                base_url=public_deepseek_base_url,timeout=120.0)
public_deepseek_client_async = AsyncOpenAI(api_key=public_deepseek_api_key,
                                base_url=public_deepseek_base_url,timeout=120.0)
public_deepseek_model = "deepseek_v31"

dashscope_api_key = os.getenv("DASHSCOPE_API_KEY")
dashscope_base_url = os.getenv("DASHSCOPE_BASE_URL")
dashscope_client = OpenAI(
    api_key=dashscope_api_key,
    base_url=dashscope_base_url,timeout=120.0)
dashscope_client_async = AsyncOpenAI(
    api_key=dashscope_api_key,
    base_url=dashscope_base_url,timeout=120.0)
qwen3_vl_32b_model = "qwen3-vl-32b-instruct"
qwen3_vl_235b_model = "qwen3-vl-235b-a22b-instruct"
qwen3_235b_model = "qwen3-235b-a22b-instruct-2507"

google_api_key = os.getenv("GOOGLE_API_KEY")
google_base_url = os.getenv("GOOGLE_BASE_URL")
google_client = OpenAI(api_key=google_api_key,
                            base_url=google_base_url,timeout=120.0)
google_client_async = AsyncOpenAI(api_key=google_api_key,
                                  base_url=google_base_url,timeout=120.0)
gemini_3_flash_model = "gemini-3-flash-preview"
gemini_3_pro_model = "gemini-3-pro-preview"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key,timeout=120.0)
openai_client_async = AsyncOpenAI(api_key=openai_api_key,timeout=120.0)
gpt_5_2_model = "gpt-5.2"
gpt_5_1_model = "gpt-5.1"
gpt_5_model = "gpt-5"
gpt_4o_model = "gpt-4o"

chatanywhere_key_backup = os.getenv("CHATANYWHERE_KEY_BACKUP")
chatanywhere_key = os.getenv("CHATANYWHERE_KEY")
chatanywhere_base_url = os.getenv("CHATANYWHERE_BASE_URL")
chatanywhere_client = OpenAI(
    api_key=chatanywhere_key,
    base_url=chatanywhere_base_url,timeout=120.0)
chatanywhere_client_async = AsyncOpenAI(
    api_key=chatanywhere_key,
    base_url=chatanywhere_base_url,timeout=120.0)

chatanywhere_client_backup = OpenAI(
    api_key=chatanywhere_key_backup,
    base_url=chatanywhere_base_url, timeout=120.0)
chatanywhere_client_async_backup = AsyncOpenAI(
    api_key=chatanywhere_key_backup,
    base_url=chatanywhere_base_url, timeout=120.0)

chatanywhere_gpt_5_2_model = "gpt-5.2"
chatanywhere_gpt_5_1_model = "gpt-5.1"
chatanywhere_gpt_5_mini_model = "gpt-5-mini"
chatanywhere_gpt_5_model = "gpt-5"
chatanywhere_gpt_4o_model = "gpt-4o"
chatanywhere_gemini_3_flash_model = "gemini-3-flash-preview"
chatanywhere_gemini_3_pro_model = "gemini-3-pro-preview"
chatanywhere_gemini_2_5_flash_model = "gemini-2.5-flash"
chatanywhere_claude_sonnet_4_model = "claude-sonnet-4-20250514"
chatanywhere_deepseek_model = "deepseek-v3.2"
chatanywhere_qwen3_235b_model = "qwen3-235b-a22b-instruct-2507"
chatanywhere_gemini_2_5_pro_model = "gemini-2.5-pro"
chatanywhere_claude_opus_model = "claude-opus-4-6"
chatanywhere_gpt_5_nano_model = "gpt-5-nano"
chatanywhere_gemini_3_1_flash_lite_model = "gemini-3.1-flash-lite-preview"


aihubmix_api_key = os.getenv("AIHUBMIX_API_KEY")
aihubmix_base_url = os.getenv("AIHUBMIX_BASE_URL")
aihubmix_client = OpenAI(
    api_key=aihubmix_api_key,
    base_url=aihubmix_base_url, timeout=120.0)
aihubmix_client_async = AsyncOpenAI(
    api_key=aihubmix_api_key,
    base_url=aihubmix_base_url, timeout=120.0)
aihubmix_gemini_free_model = "gemini-3-flash-preview-free"


deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek_base_url = os.getenv("DEEPSEEK_BASE_URL")
deepseek_client = OpenAI(
    api_key=deepseek_api_key,
    base_url=deepseek_base_url,timeout=120.0)
deepseek_client_async = AsyncOpenAI(
    api_key=deepseek_api_key,
    base_url=deepseek_base_url,timeout=120.0)
deepseek_chat_model = "deepseek-chat"
my_deepseek_model_name = 'DeepSeekV3.2'

sf_api_key = os.getenv("SF_API_KEY")
sf_base_url = os.getenv("SF_BASE_URL")
sf_client = OpenAI(
    api_key=sf_api_key,
    base_url=sf_base_url,timeout=120.0)
sf_client_async = AsyncOpenAI(
    api_key=sf_api_key,
    base_url=sf_base_url,timeout=120.0)
siliconflow_qwen3_vl_32b_model = "Qwen/Qwen3-VL-32B-Instruct"
siliconflow_qwen3_vl_235b_model = "Qwen/Qwen3-VL-235B-A22B-Instruct"

api_data = {
    "async": {
        "qwen3-vl-32b": {
            "client": dashscope_client_async,
            "model": qwen3_vl_32b_model
        },
        "gpt-5-nano-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_gpt_5_nano_model
        },
        "gemini-3-flash-free-aihub": {
            "client": aihubmix_client_async,
            "model": aihubmix_gemini_free_model
        },
        "gpt-5-mini-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_gpt_5_mini_model
        },
        "claude-opus-4-6-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_claude_opus_model
        },
        "gemini-3.1-flash-lite-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_gemini_3_1_flash_lite_model
        },
        "gemini-2.5-pro-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_gemini_2_5_pro_model
        },
        "my-deepseek": {
            "client":
            deepseek_client_async,  #  sync  deepseek_client
            "model": my_deepseek_model_name
        },
        "qwen3-235b": {
            "client": dashscope_client_async,
            "model": qwen3_235b_model
        },
        "qwen3-vl-235b": {
            "client": dashscope_client_async,
            "model": qwen3_vl_235b_model
        },
        "qwen3-vl-32b-sf": {
            "client": sf_client_async,
            "model": siliconflow_qwen3_vl_32b_model
        },
        "qwen3-vl-235b-sf": {
            "client": sf_client_async,
            "model": siliconflow_qwen3_vl_235b_model
        },
        "deepseek-v3.1-public": {
            "client": public_deepseek_client_async,
            "model": public_deepseek_model
        },
        "gemini-3-flash": {
            "client": google_client_async,
            "model": gemini_3_flash_model
        },
        "gemini-3-flash-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_gemini_3_flash_model
        },
        "gemini-3-pro-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_gemini_3_pro_model
        },
        "gemini-2.5-flash-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_gemini_2_5_flash_model
        },
        "claude-sonnet-4-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_claude_sonnet_4_model
        },
        "gpt-4o-ca": {
            "client": chatanywhere_client_async,
            "model": gpt_4o_model
        },
        "gpt-5.2-ca": {
            "client": chatanywhere_client_async,
            "model": gpt_5_2_model
        },
        "gpt-5.1-ca": {
            "client": chatanywhere_client_async,
            "model": gpt_5_1_model
        },
        "gpt-5-ca": {
            "client": chatanywhere_client_async,
            "model": gpt_5_model
        },
        "deepseek": {
            "client": deepseek_client_async,
            "model": deepseek_chat_model
        },
        "deepseek-v3.2-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_deepseek_model
        },
        "qwen3-235b-ca": {
            "client": chatanywhere_client_async,
            "model": chatanywhere_qwen3_235b_model
        }
    },
    "sync": {
        "qwen3-vl-32b": {
            "client": dashscope_client,
            "model": qwen3_vl_32b_model
        },
        "gpt-5-nano-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_gpt_5_nano_model
        },
        "gpt-5-mini-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_gpt_5_mini_model
        },
        "gemini-3.1-flash-lite-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_gemini_3_1_flash_lite_model
        },
        "gemini-3-flash-free-aihub": {
            "client": aihubmix_client,
            "model": aihubmix_gemini_free_model
        },
        "claude-opus-4-6-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_claude_opus_model
        },
        "gemini-2.5-pro-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_gemini_2_5_pro_model
        },
        "my-deepseek": {
            "client": deepseek_client,  #  sync  deepseek_client
            "model": my_deepseek_model_name
        },
        "qwen3-235b": {
            "client": dashscope_client,
            "model": qwen3_235b_model
        },
        "qwen3-vl-235b": {
            "client": dashscope_client,
            "model": qwen3_vl_235b_model
        },
        "qwen3-vl-32b-sf": {
            "client": sf_client,
            "model": siliconflow_qwen3_vl_32b_model
        },
        "qwen3-vl-235b-sf": {
            "client": sf_client,
            "model": siliconflow_qwen3_vl_235b_model
        },
        "deepseek-v3.1-public": {
            "client": public_deepseek_client,
            "model": public_deepseek_model
        },
        "gemini-3-flash": {
            "client": google_client,
            "model": gemini_3_flash_model
        },
        "gemini-3-flash-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_gemini_3_flash_model
        },
        "gemini-3-pro-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_gemini_3_pro_model
        },
        "gemini-2.5-flash-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_gemini_2_5_flash_model
        },
        "claude-sonnet-4-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_claude_sonnet_4_model
        },
        "gpt-4o-ca": {
            "client": chatanywhere_client,
            "model": gpt_4o_model
        },
        "gpt-5.2-ca": {
            "client": chatanywhere_client,
            "model": gpt_5_2_model
        },
        "gpt-5.1-ca": {
            "client": chatanywhere_client,
            "model": gpt_5_1_model
        },
        "gpt-5-ca": {
            "client": chatanywhere_client,
            "model": gpt_5_model
        },
        "deepseek": {
            "client": deepseek_client,
            "model": deepseek_chat_model
        },
        "deepseek-v3.2-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_deepseek_model
        },
        "qwen3-235b-ca": {
            "client": chatanywhere_client,
            "model": chatanywhere_qwen3_235b_model
        }
    }
}

max_tokens = {
    qwen3_vl_32b_model: 8192,
    qwen3_vl_235b_model: 8192,
    qwen3_235b_model: 8192,
    gemini_3_flash_model: 32768,
    gpt_5_2_model: 8192,
    gpt_5_1_model: 8192,
    gpt_5_model: 8192,
    chatanywhere_gemini_3_flash_model: 32768,
    chatanywhere_gemini_3_pro_model: 32768,
    chatanywhere_gemini_2_5_flash_model: 32768,
    chatanywhere_gemini_3_1_flash_lite_model: 32768,
    chatanywhere_claude_sonnet_4_model: 8192,
    chatanywhere_gpt_4o_model: 8192,
    chatanywhere_gpt_5_2_model: 8192,
    chatanywhere_gpt_5_nano_model: 8192,
    aihubmix_gemini_free_model: 32768,
    chatanywhere_gpt_5_1_model: 8192,
    chatanywhere_claude_opus_model: 8192,
    chatanywhere_gpt_5_model: 8192,
    chatanywhere_gpt_5_mini_model: 8192,
    deepseek_chat_model: 8192,
    chatanywhere_gemini_2_5_pro_model: 32768,
    chatanywhere_deepseek_model: 8192,
    chatanywhere_qwen3_235b_model: 65536,
}

async def get_response_async(prev_messages,
                             next_content,
                             model,
                             client,
                             tools=None,
                             max_retries=3,
                             MAX_TOKENS_LIMIT=0):

    # --- 60/5 = 12---
    if "free" in model.lower():
        import asyncio
        print("--- [Rate Limit Control]  12.5 ...")
        await asyncio.sleep(12.5)
    # -------------------------------------------------------------

    if MAX_TOKENS_LIMIT == 0:
        MAX_TOKENS_LIMIT = max_tokens.get(model, 8192)

    if isinstance(next_content, str):
        user_content = next_content
    else:
        user_content = next_content


    messages =  prev_messages + [{"role": "user", "content": user_content}]

    for attempt in range(max_retries):
        try:
            reasoning_content = ""
            answer_content = ""
            tool_info = []
            is_answering = False

            if tools is not None:
                response = await client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=tools,
                    parallel_tool_calls=True,
                    stream=True,
                    max_tokens=MAX_TOKENS_LIMIT

                )
            else:
                response = await client.chat.completions.create(
                    model=model,
                    messages=messages,
                    stream=True,
                    max_tokens=MAX_TOKENS_LIMIT

                )

            async for chunk in response:

                if chunk.choices:
                    delta = chunk.choices[0].delta
                    if hasattr(delta, 'reasoning_content'
                               ) and delta.reasoning_content != None:
                        reasoning_content += delta.reasoning_content
                    else:
                        if not is_answering:
                            is_answering = True
                        if delta.content is not None:
                            answer_content += delta.content
                        if delta.tool_calls is not None:

                            for tool_call in delta.tool_calls:
                                index = tool_call.index
                                while len(tool_info) <= index:
                                    tool_info.append({})
                                if tool_call.id:
                                    tool_info[
                                        index]['id'] = tool_info[index].get(
                                            'id', '') + tool_call.id
                                if tool_call.function and tool_call.function.name:
                                    tool_info[index][
                                        'name'] = tool_info[index].get(
                                            'name',
                                            '') + tool_call.function.name
                                if tool_call.function and tool_call.function.arguments:
                                    tool_info[index][
                                        'arguments'] = tool_info[index].get(
                                            'arguments',
                                            '') + tool_call.function.arguments
                                if tool_call.type:
                                    tool_info[index]['type'] = tool_call.type

            if not reasoning_content:
                if answer_content.startswith("<think>"):
                    end_think_idx = answer_content.find("</think>")
                    if end_think_idx != -1:
                        reasoning_content = answer_content[len("<think>"
                                                               ):end_think_idx]
                        answer_content = answer_content[end_think_idx +
                                                        len("</think>"):]

            new_message = {
                "role": "assistant",
                "content": answer_content,
            }
            if len(tool_info) > 0:
                tool_calls = [{
                    "id": tool_call["id"],
                    "function": {
                        "name": tool_call["name"],
                        "arguments": tool_call["arguments"]
                    },
                    "type": tool_call["type"],
                    "index": i
                } for i, tool_call in enumerate(tool_info)]
                new_message["tool_calls"] = tool_calls
            messages.append(new_message)

            return {
                "content": answer_content,
                "reasoning_content": reasoning_content,
                "usage": None,
                "prev_messages": messages,
                "tool_info": tool_info
            }

        except (APIConnectionError, InternalServerError) as e:
            print(
                f"--- [Retryable Error] (Attempt {attempt + 1}/{max_retries}): {e}"
            )
            if attempt == max_retries - 1: raise e

        except Exception as e:
            error_str = str(e).lower()

            # --- / CA  Client ---
            if "insufficient_quota" in error_str or "balance" in error_str or "402" in error_str or "quota" in error_str:
                if "chatanywhere" in str(client.base_url).lower():
                    print(f"--- [Quota Error] ChatAnywhere  Key (Attempt {attempt + 1}/{max_retries})")
                    client = chatanywhere_client_async_backup #  Async Client 
                    continue #  client
            # ----------------------------------------------------------------

            if "incomplete chunked read" in error_str or "peer closed connection" in error_str or "connection closed" in error_str:
                print(
                    f"--- [Network/Server Cutoff] (Attempt {attempt + 1}/{max_retries}): {e}"
                )
                if attempt == max_retries - 1:
                    print("--- Max retries reached for cutoff error.")
                    raise e
                print(
                    "--- Server likely overloaded. Sleeping for 10 seconds...")
            else:
                print(f"--- [Fatal Error]: {e}")
                raise e

def get_response(prev_messages,
                 next_content,
                 model,
                 client,
                 tools=None,
                 max_retries=3,
                 MAX_TOKENS_LIMIT=0):

    if MAX_TOKENS_LIMIT == 0:
        MAX_TOKENS_LIMIT = max_tokens.get(model, 8192)

    # ---  ---
    if "free" in model.lower():
        import time
        print("--- [Rate Limit Control]  12.5 ...")
        time.sleep(12.5)
    # ----------------------------------------

    if isinstance(next_content, str):
        user_content = next_content
    else:
        user_content = next_content

    # ---  System Prompt Token ---
    system_messages = []
    #  gpt-5-nano  aihub 
    if model in ["gpt-5-nano-ca", "gemini-3-flash-preview-free"]:
        system_messages = [{
            "role": "system",
            "content": "CRITICAL INSTRUCTION: DO NOT output any step-by-step reasoning, thinking processes, or <think> tags. Provide ONLY the final result strictly in the requested format (like JSON). Be as concise as possible."
        }]

    messages = system_messages + prev_messages + [{"role": "user", "content": user_content}]

    for attempt in range(max_retries):
        try:
            reasoning_content = ""
            answer_content = ""
            tool_info = []
            is_answering = False

            if tools is not None:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=tools,
                    parallel_tool_calls=True,
                    stream=True,
                    max_tokens=MAX_TOKENS_LIMIT
                )
            else:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    stream=True,
                    max_tokens=MAX_TOKENS_LIMIT
                )

            for chunk in response:
                if chunk.choices:
                    delta = chunk.choices[0].delta
                    if hasattr(delta, 'reasoning_content'
                               ) and delta.reasoning_content != None:
                        reasoning_content += delta.reasoning_content
                    else:
                        if not is_answering:
                            is_answering = True
                        if delta.content is not None:
                            answer_content += delta.content
                        if delta.tool_calls is not None:
                            for tool_call in delta.tool_calls:
                                index = tool_call.index
                                while len(tool_info) <= index:
                                    tool_info.append({})
                                if tool_call.id:
                                    tool_info[
                                        index]['id'] = tool_info[index].get(
                                            'id', '') + tool_call.id
                                if tool_call.function and tool_call.function.name:
                                    tool_info[index][
                                        'name'] = tool_info[index].get(
                                            'name',
                                            '') + tool_call.function.name
                                if tool_call.function and tool_call.function.arguments:
                                    tool_info[index][
                                        'arguments'] = tool_info[index].get(
                                            'arguments',
                                            '') + tool_call.function.arguments
                                if tool_call.type:
                                    tool_info[index]['type'] = tool_call.type

            if not reasoning_content:
                if answer_content.startswith("<think>"):
                    end_think_idx = answer_content.find("</think>")
                    if end_think_idx != -1:
                        reasoning_content = answer_content[len("<think>"
                                                               ):end_think_idx]
                        answer_content = answer_content[end_think_idx +
                                                        len("</think>"):]

            new_message = {
                "role": "assistant",
                "content": answer_content,
            }
            if len(tool_info) > 0:
                tool_calls = [{
                    "id": tool_call["id"],
                    "function": {
                        "name": tool_call["name"],
                        "arguments": tool_call["arguments"]
                    },
                    "type": tool_call["type"],
                    "index": i
                } for i, tool_call in enumerate(tool_info)]
                new_message["tool_calls"] = tool_calls
            messages.append(new_message)
            return {
                "content": answer_content,
                "reasoning_content": reasoning_content,
                "usage": None,
                "prev_messages": messages,
                "tool_info": tool_info
            }
        except (APIConnectionError, InternalServerError) as e:
            print(
                f"--- [Retryable Error] (Attempt {attempt + 1}/{max_retries}): {e}"
            )
            if attempt == max_retries - 1: raise e
            time.sleep(5)
        except Exception as e:
            error_str = str(e).lower()

            # --- / CA  Client ---
            if "insufficient_quota" in error_str or "balance" in error_str or "402" in error_str or "quota" in error_str:
                if "chatanywhere" in str(client.base_url).lower():
                    print(f"--- [Quota Error] ChatAnywhere  Key (Attempt {attempt + 1}/{max_retries})")
                    client = chatanywhere_client_backup #  Sync Client 
                    continue
            # ---------------------------------------------------------------

            if "incomplete chunked read" in error_str or "peer closed connection" in error_str or "connection closed" in error_str:
                print(
                    f"--- [Network/Server Cutoff] (Attempt {attempt + 1}/{max_retries}): {e}"
                )
                if attempt == max_retries - 1:
                    print("--- Max retries reached for cutoff error.")
                    raise e
                print(
                    "--- Server likely overloaded. Sleeping for 10 seconds...")
                time.sleep(10)
            else:
                print(f"--- [Fatal Error]: {e}")
                raise e

def pack_content(prompt, images):
    image_list = images or []
    content = [
        {"type": "image_url", "image_url": img_url}
        for img_url in image_list
    ] + [
        {"type": "text", "text": prompt}
    ]
    return content

def openai_pack_content(prompt, images):
    image_list = images or []
    content = [
        {"type": "image_url", "image_url": {
            "url": img_url,
            "detail": "auto"
        }}
        for img_url in image_list
    ] + [
        {"type": "text", "text": prompt}
    ]
    return content

def process_output(response_content):
    try:
        if "```json" in response_content:
            start_idx = response_content.index("```json") + len("```json")
            res_content = response_content[start_idx:].lstrip()

            if "```" in res_content:
                end_idx = res_content.index("```")
                json_str = res_content[:end_idx].strip()
            else:
                if "}" not in res_content:
                    res_content += "}"
                end_idx = res_content.rindex("}")
                json_str = res_content[:end_idx + 1].strip()

            return json.loads(json_str)
        elif "{" in response_content:
            start_idx = response_content.index("{")
            if "}" not in response_content:
                response_content += "}"
            end_idx = response_content.rindex("}")
            json_str = response_content[start_idx:end_idx + 1].strip()

            return json.loads(json_str)
        else:
            return json.loads(response_content)
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {e}")
        with open("debug_response.txt", "a") as f:
            f.write("-----------------------\n")
            f.write(f"Failed to decode JSON from response:\n{response_content}\n")
        return None
    except Exception as e:
        print(f"Unexpected Error: {e}")
        with open("debug_response.txt", "a") as f:
            f.write("-----------------------\n")
            f.write(f"Unexpected error processing response:\n{response_content}\n")
        return None

In [ ]:
def test_model(model):
    print(f"Testing model: {model}")
    client_info = api_data["sync"].get(model)
    if not client_info:
        print(f"No client info found for model: {model}")
        return
    client = client_info["client"]
    model_name = client_info["model"]
    messages = []
    prompt = "Who are you?"
    response = get_response(messages, prompt, model_name, client)
    print(f"Response from model {model}:\n{response['content']}")

In [ ]:

# 1.  ChatAnywhere  Gemini 2.5 Pro
test_model("gemini-2.5-pro-ca")

print("-" * 50)  # 

# 2.  DeepSeek v3.2
#test_model("gpt-5-mini-ca")

In [ ]:
output1 = "```json\n{\n  \"answer\": \"The capital of France is Paris.\"\n}```"
output2 = "```json\n{\n  \"answer\": \"The capital of France is Paris.\"\n}\n"
output3 = "{\n  \"answer\": \"The capital of France is Paris.\"\n}\n"
output4 = "Here is the answer:\n```json\n{\n  \"answer\": \"The capital of France is Paris.\"\n}\n```"
output5 = "The answer is:```json{\n  \"answer\": \"The capital of France is Paris.\"\n}```"
output6 = "{}aaa"
output7 = "```json{\n  \"answer\": \"The capital of France is Paris.\"\n}\n```"
output8 = "```json\n{\n  \"answer\": \"The capital of France is Paris.\"\n}```"
print("1")
print(process_output(output1))
print("2")
print(process_output(output2))
print("3")
print(process_output(output3))
print("4")
print(process_output(output4))
print("5")
print(process_output(output5))
print("6")
print(process_output(output6))
print("7")
print(process_output(output7))
print("8")
print(process_output(output8))

In [ ]:
LLM_OPEN_ANSWER_TEMPLATE = """
You are a knowledgeable student taking a biomedical examination. You will be provided with a question and corresponding image(s).
You need to provide a detailed reasoning process leading to your final answer.
```

Question:
{question}

Your Answer:
"""

# Use Logic_Chain instead of Reference_Answer
LLM_OPEN_JUDGE_TEMPLATE = """
Role: Expert Biomedical Evaluator.
Task: Evaluate the quality of a [Student_Answer] strictly against the [Logic_Chain].



1. Experiments Evaluation (Iterate through every item in the "experiments" list of the Logic_Chain):

### Evaluation Rules (Strict Binary Checklist)
Parse the [Logic_Chain] JSON and evaluate the [Student_Answer] against each specific field. Output 1 for Correct/Present, 0 for Incorrect/Missing.

   - visual_phenomenon: Did the student identify this specific all key visual feature mentioned in the logic chain?
      - In case of original visual_phenomenon marked as [Missing] in Logic Chain, the student does not need to do anything here, and you should give a -1 score as a special mark.
   - interpretation: Did the student provide the correct medical interpretation of the visual phenomenon as per the logic chain?
   - sub-conclusion: Did the student reach the correct intermediate conclusion based on the visual phenomenon and its interpretation?

2. Conclusion Evaluation:

### Conclusion Scoring Rubric (Conclusion Accuracy 0-4)
Evaluate ONLY the quality of the Student's final conclusion compared strictly against the Logic_Chain's "conclusion".
- 4 (Perfect Match): The student's conclusion identifies the exact diagnosis or result as defined in the Logic_Chain's "conclusion".
- 3 (High Accuracy): Correct diagnosis, captures ALL key points from the reference; but misses non-critical qualifiers.
- 2 (Partial/General): Identifies the correct general category or main disease.
- 1 (Vague/Weak): The conclusion is ambiguous or barely touches the truth.
- 0 (Mismatch): Wrong diagnosis, contradicts the Logic_Chain, or hallucinated conclusion.

### Input Data
1. Question: {question}
2. Logic_Chain: {logic_chain}
3. Student_Answer: {student_answer}


### Output Format
Return ONLY a strictly valid JSON object. The "experiments" array must have the EXACT SAME number of items as the input Logic_Chain.

{{
  "short_comment": "Concise justification for the 0/1 markings, specifically pointing out missing evidence or logic errors.",
  "experiments": [
    {{
      "visual_phenomenon": 0 or 1 or -1,
      "interpretation": 0 or 1,
      "sub-conclusion": 0 or 1
    }},
    ... (Repeat strictly for each item in the input Logic_Chain experiments list)
  ],
  "conclusion_score": <integer 0-4, representing the accuracy of the final conclusion ONLY>,
}}
"""

LLM_MULTIPLE_CHOICE_ANSWER_TEMPLATE = """Not applicable"""
LLM_MULTIPLE_CHOICE_JUDGE_TEMPLATE = """Not applicable"""

In [ ]:
import json

metadata_file = "./all_original_metadata_10k_with_images.json"

with open(metadata_file, 'r') as f:
    all_metadata = json.load(f)

In [ ]:
from PIL import Image

def get_images_from_folder(sample, difficulty):
    image_root = "./temp_images/"
    original_sample_index = sample["original_sample_index"]
    image_folder = os.path.join(image_root, f"sample_{original_sample_index}")
    if difficulty == "basic":
        image_indices = sample["basic_qa"]["image_indices"]
    else:
        image_indices = sample["hard_qa"]["image_indices"]
    image_paths = []
    for idx in image_indices:
        image_path = os.path.join(image_folder, f"image_{idx}.jpeg")
        image_paths.append(image_path)
    images = []
    for img_path in image_paths:
        if not os.path.exists(img_path):
            pass
            # print(f"Image not found: {img_path}")
        else:
            img = Image.open(img_path).convert("RGB")
            images.append(img)
    # to url
    image_urls = []
    for img in images:
        buffered = BytesIO()
        img.save(buffered, format="JPEG")
        img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
        image_urls.append(f"data:image/jpeg;base64,{img_str}")
    return image_urls   

def get_images_from_metadata(sample, difficulty):
    # if difficulty == "basic":
    #     image_indices = sample["basic_qa"]["image_indices"]
    # else:
    #     image_indices = sample["hard_qa"]["image_indices"]
    original_sample_index = sample["original_sample_index"]
    original_sample_index = str(original_sample_index)
    sample_metadata = all_metadata[original_sample_index]
    sample_image_info = sample_metadata["image_info"]
    # image_base64_list = []
    # # print(original_sample_index, len(sample_image_info), image_indices)
    # for idx in image_indices:
    #     if idx - 1 >= len(sample_image_info):
    #         continue
    #     img_info = sample_image_info[idx - 1]
    #     img_base64 = img_info["image_base64"]
    #     image_base64_list.append(img_base64)
    image_base64_list = [img_info["image_base64"] for img_info in sample_image_info]
    image_urls = [f"data:image/jpeg;base64,{img_base64}" for img_base64 in image_base64_list]
    return image_urls

def get_images(sample, difficulty, source="metadata"):
    if source == "folder": # Do not use
        return get_images_from_folder(sample, difficulty)
    elif source == "metadata":
        return get_images_from_metadata(sample, difficulty)
    elif source == "none":
        return []
    else:
        return []

def get_question(sample, question_type, difficulty):
    if question_type == "open":
        if difficulty == "basic":
            return sample["basic_qa"]["question"]
        else:
            return sample["hard_qa"]["question"]
    elif question_type == "mc":
        if difficulty == "basic":
            return sample["basic_qa"]["question"] + "\nChoices: \n" + "\n".join(sample["basic_qa"]["choices"])
        else:
            return sample["hard_qa"]["question"] + "\nChoices: \n" + "\n".join(sample["hard_qa"]["choices"])
        
def get_reference_answer(sample, question_type, difficulty):
    if question_type == "open":
        if difficulty == "basic":
            return sample["basic_qa"]["answer"]
        else:
            return sample["hard_qa"]["answer"]
    elif question_type == "mc":
        if difficulty == "basic":
            return sample["basic_qa"]["answer"]
        else:
            return sample["hard_qa"]["answer"]

In [ ]:
def llm_answer(question, images, question_type, model, client):
    if question_type == "open":
        answer_template = LLM_OPEN_ANSWER_TEMPLATE
    elif question_type == "mc":
        answer_template = LLM_MULTIPLE_CHOICE_ANSWER_TEMPLATE
    prompt = answer_template.format(question=question)
    content = openai_pack_content(prompt, images)
    messages = []
    response = get_response(messages, content, model, client)
    return response

In [ ]:
def llm_judge(question, reference_answer, student_answer, logic_chain, images, question_type, model, client):
    if question_type == "open":
        judge_template = LLM_OPEN_JUDGE_TEMPLATE
    elif question_type == "mc":
        judge_template = LLM_MULTIPLE_CHOICE_JUDGE_TEMPLATE
    prompt = judge_template.format(
        question=question,
        reference_answer=reference_answer,
        student_answer=student_answer,
        logic_chain = logic_chain
    )
    messages = []
    if images:
        content = openai_pack_content(prompt, images)
    else:
        content = prompt
    messages = []
    response = get_response(messages, content, model, client)
    return response

In [ ]:
async def llm_answer_async(question, images, question_type, model, client):
    if question_type == "open":
        answer_template = LLM_OPEN_ANSWER_TEMPLATE
    elif question_type == "mc":
        answer_template = LLM_MULTIPLE_CHOICE_ANSWER_TEMPLATE
    prompt = answer_template.format(question=question)
    content = openai_pack_content(prompt, images)
    messages = []
    response = await get_response_async(messages, content, model, client, max_retries=3)
    return response

async def llm_judge_async(question, reference_answer, logic_chain, student_answer, images, question_type, model, client):
    if question_type == "open":
        judge_template = LLM_OPEN_JUDGE_TEMPLATE
    elif question_type == "mc":
        judge_template = LLM_MULTIPLE_CHOICE_JUDGE_TEMPLATE
    prompt = judge_template.format(
        question=question,
        #reference_answer=,
        student_answer=student_answer,
        # student_answer = reference_answer,
        logic_chain=logic_chain
    )
    messages = []
    if images:
        content = openai_pack_content(prompt, images)
    else:
        content = prompt
    messages = []
    response = await get_response_async(messages, content, model, client)
    return response

In [ ]:
import asyncio
import json
import time
from tqdm.asyncio import tqdm as tqdm_async  # Recommended for async progress bars
from aiofiles import open as aio_open

CONCURRENCY_LIMIT = 20
LOCAL_LIMIT = 32

# Semaphore to control max concurrent connections
sem = {}
for api_key in api_data["async"].keys():
    if "local" in api_key:
        sem[api_key] = asyncio.Semaphore(LOCAL_LIMIT)
    # ---  1 ---
    elif "free" in api_key.lower():
        sem[api_key] = asyncio.Semaphore(1)
    # ----------------------------------------------
    else:
        sem[api_key] = asyncio.Semaphore(CONCURRENCY_LIMIT)

# # Configuration
# 
# answer_model = "lingshu"
# judge_model = "deepseek"
# 
# difficulty = "basic" # "basic" or "hard"
# question_type = "open" # "open" or "mc"
# 
# data_name = "batch_0_open" # file name without extension

In [ ]:
# with open(f"{data_name}.json", "r") as f:
#     data_under_judge = json.load(f)


In [ ]:
def get_answer_cache_dir_name(question_type, difficulty, answer_model, data_name, tag=""):
    return f"./cache/{answer_model}_{question_type}_{difficulty}_{data_name}{('_' + tag) if tag else ''}/"

def get_answer_file_name(question_type, difficulty, answer_model, data_name, tag=""):
    return f"llm_answers_{question_type}_{difficulty}_{answer_model}_{data_name}{('_' + tag) if tag else ''}.json"

async def process_single_answer(idx, item, question_type, model_name, client, answer_model, difficulty, image_source, cache_dir=""):
    """
    Handles the logic for a single sample: retries, answering, and processing.
    """

    if cache_dir:
        cache_file_path = os.path.join(cache_dir, f"answer_{idx}.json")
        if os.path.exists(cache_file_path):
            async with aio_open(cache_file_path, "r") as f:
                cached_content = await f.read()
                cached_result = json.loads(cached_content)
                if cached_result:
                    return cached_result
            
    question = get_question(item, question_type, difficulty)
    images = get_images(item, difficulty, source=image_source)
    ground_truth = get_reference_answer(item, question_type, difficulty)
    
    max_tries = 3
    answer = None

    # We use the semaphore here to limit how many tasks enter the 'active' state
    async with sem[answer_model]:
        while max_tries > 0:
            max_tries -= 1
            try:
                # NOTE: You must ensure 'llm_answer' is async or wrap it.
                # If your current llm_answer is sync, you need to rewrite it to use 'await' 
                # with the async client.
                answer_result = await llm_answer_async(
                    question,
                    images,
                    question_type,
                    model_name,
                    client
                )
                
                # Assuming process_output is purely CPU bound (sync), that's fine.
                # If it involves IO, make it async too.
                # answer = process_output(answer_result["content"])
                answer = answer_result["content"]
                
                if not answer:
                    print("Failed to parse answer, retrying...")
                    # Non-blocking sleep
                    await asyncio.sleep(1)
                    continue
                
                # Success
                break
            except Exception as e:
                print(f"Error: {e}") # Optional: reduce noise in async logs
                await asyncio.sleep(1)
    
    # Return the structured result
    result = {
        "question_index": idx,
        "question": question,
        "model_answer": answer,
        "ground_truth": ground_truth
    }

    if cache_dir and result["model_answer"]:
        async with aio_open(cache_file_path, "w") as f:
            await f.write(json.dumps(result, indent=2, ensure_ascii=False))

    return result

async def answer_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source, tag="", position=0):
    # 1. Get Async Client and Model Name
    # Ensure api_data["async"] is structured correctly
    if image_source == "none":
        model_name = api_data["async"][answer_model]["model"]
        client = api_data["async"][answer_model]["client"]
    else:
        model_name = api_data["async"][answer_model]["model"]
        client = api_data["async"][answer_model]["client"]

    cache_dir = get_answer_cache_dir_name(question_type, difficulty, answer_model, data_name, tag)
    os.makedirs(cache_dir, exist_ok=True)
    # 2. Create Tasks
    tasks = []
    for idx, item in enumerate(data_under_judge):
        task = process_single_answer(idx, item, question_type, model_name, client, answer_model, difficulty, image_source, cache_dir)
        tasks.append(task)

    # 3. Run Tasks with Progress Bar
    # tqdm_async.gather works like asyncio.gather but shows a progress bar
    results = await tqdm_async.gather(*tasks, desc = f"Answering with {answer_model}", position=position)

    # 4. Filter out any totally failed results (optional, depending on preference)
    # results = [r for r in results if r["model_answer"] is not None]

    # 5. Save Results
    output_filename = get_answer_file_name(question_type, difficulty, answer_model, data_name, tag)
    async with aio_open(output_filename, "w") as f:
        await f.write(json.dumps(results, indent=2, ensure_ascii=False))
    
    print(f"Saved {len(results)} answers to {output_filename}")



In [ ]:
def get_answer_cache_dir_name_half(question_type, difficulty, answer_model, data_name, tag=""):
    return f"./cache/{answer_model}_{question_type}_{difficulty}_{data_name}{('_' + tag) if tag else ''}/"

def get_answer_file_name_half(question_type, difficulty, answer_model, data_name, tag=""):
    return f"llm_answers_{question_type}_{difficulty}_{answer_model}_{data_name}{('_' + tag) if tag else ''}.json"

def get_images_half(sample, difficulty, source="metadata"):
    if source == "none":
        return []
    original_sample_index = sample["original_sample_index"]
    original_sample_index = str(original_sample_index)
    sample_metadata = all_metadata[original_sample_index]
    sample_image_info = sample_metadata["image_info"]
    # image_base64_list = []
    # # print(original_sample_index, len(sample_image_info), image_indices)
    # for idx in image_indices:
    #     if idx - 1 >= len(sample_image_info):
    #         continue
    #     img_info = sample_image_info[idx - 1]
    #     img_base64 = img_info["image_base64"]
    #     image_base64_list.append(img_base64)
    image_base64_list = [img_info["image_base64"] for img_info in sample_image_info]
    # load these images
    images = []
    for img_base64 in image_base64_list:
        img_data = base64.b64decode(img_base64)
        img = Image.open(BytesIO(img_data)).convert("RGB")
        images.append(img)
    # scale to 1/2 size
    scaled_images = []
    for img in images:
        width, height = img.size
        new_size = (width // 2, height // 2)
        scaled_img = img.resize(new_size)
        buffered = BytesIO()
        scaled_img.save(buffered, format="JPEG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
        scaled_images.append(img_base64)
        
    image_urls = [f"data:image/jpeg;base64,{img_base64}" for img_base64 in scaled_images]
    return image_urls

async def process_single_answer_half(idx, item, question_type, model_name, client, answer_model, difficulty, image_source, cache_dir=""):
    """
    Handles the logic for a single sample: retries, answering, and processing.
    """

    if cache_dir:
        cache_file_path = os.path.join(cache_dir, f"answer_{idx}.json")
        if os.path.exists(cache_file_path):
            async with aio_open(cache_file_path, "r") as f:
                cached_content = await f.read()
                cached_result = json.loads(cached_content)
                if cached_result:
                    return cached_result
            
    question = get_question(item, question_type, difficulty)
    images = get_images_half(item, difficulty, source=image_source)
    ground_truth = get_reference_answer(item, question_type, difficulty)
    
    max_tries = 1
    answer = None

    # We use the semaphore here to limit how many tasks enter the 'active' state
    async with sem[answer_model]:
        while max_tries > 0:
            max_tries -= 1
            try:
                # NOTE: You must ensure 'llm_answer' is async or wrap it.
                # If your current llm_answer is sync, you need to rewrite it to use 'await' 
                # with the async client.
                answer_result = await llm_answer_async(
                    question,
                    images,
                    question_type,
                    model_name,
                    client
                )
                
                # Assuming process_output is purely CPU bound (sync), that's fine.
                # If it involves IO, make it async too.
                # answer = process_output(answer_result["content"])
                answer = answer_result["content"]
                
                if not answer:
                    print("Failed to parse answer, retrying...")
                    # Non-blocking sleep
                    await asyncio.sleep(1)
                    continue
                
                # Success
                break
            except Exception as e:
                print(f"Error: {e}") # Optional: reduce noise in async logs
                await asyncio.sleep(1)
    
    # Return the structured result
    result = {
        "question_index": idx,
        "question": question,
        "model_answer": answer,
        "ground_truth": ground_truth
    }

    if cache_dir and result["model_answer"]:
        async with aio_open(cache_file_path, "w") as f:
            await f.write(json.dumps(result, indent=2, ensure_ascii=False))

    return result

async def answer_half_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source, tag="", position=0):
    # 1. Get Async Client and Model Name
    # Ensure api_data["async"] is structured correctly
    if image_source == "none":
        model_name = api_data["async"][answer_model]["model"]
        client = api_data["async"][answer_model]["client"]
    else:
        model_name = api_data["async"][answer_model]["model"]
        client = api_data["async"][answer_model]["client"]

    cache_dir = get_answer_cache_dir_name(question_type, difficulty, answer_model, data_name, tag)
    os.makedirs(cache_dir, exist_ok=True)
    # 2. Create Tasks
    tasks = []
    for idx, item in enumerate(data_under_judge):
        task = process_single_answer_half(idx, item, question_type, model_name, client, answer_model, difficulty, image_source, cache_dir)
        tasks.append(task)

    # 3. Run Tasks with Progress Bar
    # tqdm_async.gather works like asyncio.gather but shows a progress bar
    results = await tqdm_async.gather(*tasks, desc = f"Answering with {answer_model}", position=position)

    # 4. Filter out any totally failed results (optional, depending on preference)
    # results = [r for r in results if r["model_answer"] is not None]

    # 5. Save Results
    output_filename = get_answer_file_name(question_type, difficulty, answer_model, data_name, tag)
    async with aio_open(output_filename, "w") as f:
        await f.write(json.dumps(results, indent=2, ensure_ascii=False))
    
    print(f"Saved {len(results)} answers to {output_filename}")



In [ ]:
# 1/3 async version
def get_images_third(sample, difficulty, source="metadata"):
    if source == "none":
        return []
    original_sample_index = sample["original_sample_index"]
    original_sample_index = str(original_sample_index)
    sample_metadata = all_metadata[original_sample_index]
    sample_image_info = sample_metadata["image_info"]
    image_base64_list = [img_info["image_base64"] for img_info in sample_image_info]
    # load these images
    images = []
    for img_base64 in image_base64_list:
        img_data = base64.b64decode(img_base64)
        img = Image.open(BytesIO(img_data)).convert("RGB")
        images.append(img)
    # scale to 1/3 size
    scaled_images = []
    for img in images:
        width, height = img.size
        new_size = (width // 3, height // 3)
        scaled_img = img.resize(new_size)
        buffered = BytesIO()
        scaled_img.save(buffered, format="JPEG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
        scaled_images.append(img_base64)
        
    image_urls = [f"data:image/jpeg;base64,{img_base64}" for img_base64 in scaled_images]
    return image_urls

async def process_single_answer_third(idx, item, question_type, model_name, client, answer_model, difficulty, image_source, cache_dir=""):
    """
    Handles the logic for a single sample: retries, answering, and processing.
    """

    if cache_dir:
        cache_file_path = os.path.join(cache_dir, f"answer_{idx}.json")
        if os.path.exists(cache_file_path):
            async with aio_open(cache_file_path, "r") as f:
                cached_content = await f.read()
                cached_result = json.loads(cached_content)
                if cached_result:
                    return cached_result
            
    question = get_question(item, question_type, difficulty)
    images = get_images_third(item, difficulty, source=image_source)
    ground_truth = get_reference_answer(item, question_type, difficulty)
    
    max_tries = 1
    answer = None

    # We use the semaphore here to limit how many tasks enter the 'active' state
    async with sem[answer_model]:
        while max_tries > 0:
            max_tries -= 1
            try:
                answer_result = await llm_answer_async(
                    question,
                    images,
                    question_type,
                    model_name,
                    client
                )
                
                answer = answer_result["content"]
                
                if not answer:
                    print("Failed to parse answer, retrying...")
                    await asyncio.sleep(1)
                    continue
                
                break
            except Exception as e:
                print(f"Error: {e}")
                await asyncio.sleep(1)
    
    result = {
        "question_index": idx,
        "question": question,
        "model_answer": answer,
        "ground_truth": ground_truth
    }

    if cache_dir and result["model_answer"]:
        async with aio_open(cache_file_path, "w") as f:
            await f.write(json.dumps(result, indent=2, ensure_ascii=False))

    return result

async def answer_third_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source, tag="", position=0):
    if image_source == "none":
        model_name = api_data["async"][answer_model]["model"]
        client = api_data["async"][answer_model]["client"]
    else:
        model_name = api_data["async"][answer_model]["model"]
        client = api_data["async"][answer_model]["client"]

    cache_dir = get_answer_cache_dir_name(question_type, difficulty, answer_model, data_name, tag)
    os.makedirs(cache_dir, exist_ok=True)
    tasks = []
    for idx, item in enumerate(data_under_judge):
        task = process_single_answer_third(idx, item, question_type, model_name, client, answer_model, difficulty, image_source, cache_dir)
        tasks.append(task)

    results = await tqdm_async.gather(*tasks, desc = f"Answering with {answer_model}", position=position)

    output_filename = get_answer_file_name(question_type, difficulty, answer_model, data_name, tag)
    async with aio_open(output_filename, "w") as f:
        await f.write(json.dumps(results, indent=2, ensure_ascii=False))
    
    print(f"Saved {len(results)} answers to {output_filename}")

In [ ]:
# await answer_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source)

In [ ]:
import copy 

def is_valid_judgement(judgement):
    if judgement is None:
        print("Judgement is None")
        return False
    if "conclusion_score" not in judgement:
        print("Missing conclusion_score")
        return False
    conclusion_score = judgement["conclusion_score"]
    if not isinstance(conclusion_score, int):
        print("conclusion_score is not int")
        return False
    if conclusion_score < 0 or conclusion_score > 4:
        print("conclusion_score out of range")
        return False
    if "experiments" not in judgement:
        print("Missing experiments")
        return False
    if not isinstance(judgement["experiments"], list):
        print("experiments is not a list")
        return False
    for exp in judgement["experiments"]:
        if not all(key in exp for key in ["visual_phenomenon", "interpretation", "sub-conclusion"]):
            print("Missing keys in experiment")
            return False
        if not all(isinstance(exp[key], int) for key in ["visual_phenomenon", "interpretation", "sub-conclusion"]):
            print("Experiment values are not int")
            return False
        if not all(exp[key] in [0, 1, -1] for key in ["visual_phenomenon", "interpretation", "sub-conclusion"]):
            print("Experiment values out of range")
            return False
        if "interpretation" in exp and exp["interpretation"] not in [0, 1, -1]:
            print("interpretation value out of range")
            return False
        if "sub-conclusion" in exp and exp["sub-conclusion"] not in [0, 1, -1]:
            print("sub-conclusion value out of range")
            return False
    return True

def correction(judgement):
    for experiment in judgement.get("experiments", []):
        for key in ["interpretation", "sub-conclusion"]:
            if experiment.get(key) == -1:
                experiment[key] = 1

async def process_single_judgement(idx, item, answer_entry, question_type, model_name, client, judge_model, difficulty, cache_dir=""):


    if cache_dir:
        cache_file_path = os.path.join(cache_dir, f"judgement_{idx}.json")
        if os.path.exists(cache_file_path):
            async with aio_open(cache_file_path, "r") as f:
                cached_content = await f.read()
                cached_result = json.loads(cached_content)
                if cached_result:
                    is_valuable = False # cached_result["judgement"]["correctness"] < 2
                    return cached_result, is_valuable

    question = get_question(item, question_type, difficulty)
    ground_truth = get_reference_answer(item, question_type, difficulty)
    model_answer = answer_entry["model_answer"]

    input_logic_list = item.get("input_logic_chain", [])
    raw_logic_chain = input_logic_list[0] if input_logic_list and len(input_logic_list) > 0 else {}
    logic_chain_cleaned = copy.deepcopy(raw_logic_chain)
    if "experiments" in logic_chain_cleaned:
        for exp in logic_chain_cleaned["experiments"]:
            exp.pop("experimental_setting", None)
            exp.pop("experiment_goal", None)
    if "reasoning" in logic_chain_cleaned:
        logic_chain_cleaned["reasoning"].pop("intermediate_inferences", None)
    
    #logic_chain_str = json.dumps(logic_chain_cleaned, ensure_ascii=False)
    

    if model_answer is None:
        judgement_entry = {
            "question_index": idx,
            "question": question,
            "model_answer": None,
            "ground_truth": ground_truth,
            "logic_chain": raw_logic_chain,
            "judgement": {
                "experiments": [],
                "short_comment": "Model failed to give a valid answer."
            }
        }
        return judgement_entry, False


    async with sem[judge_model]:
        while True:
            try:
                
                judgement_result = await llm_judge_async(
                    question=question,
                    reference_answer=ground_truth,
                    logic_chain=logic_chain_cleaned,
                    student_answer=model_answer,
                    images=[],
                    question_type=question_type,
                    model=model_name,
                    client=client
                )
                
                judgement = process_output(judgement_result["content"])
                
                if judgement is None:
                    
                    print("Failed to parse judgement, retrying...")
                    await asyncio.sleep(1)
                    continue

                if not is_valid_judgement(judgement):
                    print("Invalid judgement format, retrying...")
                    with open("invalid_judgement_log.txt", "a") as log_f:
                        log_f.write(f"Index {idx} - Judgement: {json.dumps(judgement, ensure_ascii=False)}\n")
                    await asyncio.sleep(1)
                    continue

                correction(judgement)
                
                
                judgement_entry = {
                    "question_index": idx,
                    "question": question,
                    "model_answer": model_answer,
                    "ground_truth": ground_truth,
                    "logic_chain": raw_logic_chain,
                    "judgement": judgement
                }
                
               
                if cache_dir:
                    async with aio_open(cache_file_path, "w") as f:
                        await f.write(json.dumps(judgement_entry, indent=2, ensure_ascii=False))
                
                return judgement_entry, False #is_valuable

            except Exception as e:
                print(f"Error during judging: {e}. Retrying...")
                await asyncio.sleep(1)

def get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name, tag=""):
    return f"llm_judgements_{question_type}_{difficulty}_{answer_model}_{judge_model}_{data_name}{('_' + tag) if tag else ''}.json"

def get_valuable_file_name(question_type, difficulty, answer_model, judge_model, data_name, tag=""):
    return f"valuable_questions_{question_type}_{difficulty}_{answer_model}_{judge_model}_{data_name}{('_' + tag) if tag else ''}.json"

def get_judgement_cache_dir_name(question_type, difficulty, answer_model, judge_model, data_name, tag=""):
    return f"./cache/{answer_model}_{judge_model}_{question_type}_{difficulty}_{data_name}{('_' + tag) if tag else ''}/"

async def judgement_main(judge_model, question_type, difficulty, answer_model, data_name, data_under_judge, image_source, tag="", position=0):
    
    judge_model_name = api_data["async"][judge_model]["model"]
    judge_client = api_data["async"][judge_model]["client"]


    cache_dir = get_judgement_cache_dir_name(question_type, difficulty, answer_model, judge_model, data_name, tag)
    os.makedirs(cache_dir, exist_ok=True)
    
    
    answer_file = get_answer_file_name(question_type, difficulty, answer_model, data_name, tag)
    with open(answer_file, "r") as f:
        answers = json.load(f)
    tasks = []
    for idx, (item, answer_entry) in enumerate(zip(data_under_judge, answers)):
        task = process_single_judgement(idx, item, answer_entry, question_type, judge_model_name, judge_client, judge_model, difficulty, cache_dir)
        tasks.append(task)

    
    print(f"Starting async judgement for {len(tasks)} samples...")
    results = await tqdm_async.gather(*tasks, desc=f"Judging Samples for {answer_model} with {judge_model}", position=position)

    
    final_judgements = []
    # final_valuable_questions = []

    
    for i, (judgement_entry, is_valuable) in enumerate(results):
        final_judgements.append(judgement_entry)
        
        # if is_valuable:
        #     
        #     data_under_judge[i]["question_index"] = i 
        #     final_valuable_questions.append(data_under_judge[i])
            
    # # 1-5
    # score_distribution = {str(i): 0 for i in range(-1, 5)}
    # total_valid = 0
    # for entry in final_judgements:
    #     score = entry["judgement"]["correctness"]
    #     if str(score) in score_distribution:
    #         score_distribution[str(score)] += 1
    #         if str(score) != "-1":
    #             total_valid += 1
    # if total_valid > 0:
    #     score_percentages = {}
    #     for key, value in score_distribution.items():
    #         if key != "-1":
    #             score_percentages[key] = float(f"{value / total_valid:.4f}")
    # print(f"Score Distribution for {answer_model} judged by {judge_model}: # {score_distribution}")
    # print(f"Total Valid Answer(s): {total_valid}")
    # if total_valid > 0:
    #     print(f"Score Distribution (Percentages) for {answer_model} judged by {judge_model}: {score_percentages}")
    # print(f"Total Valuable Questions for {answer_model} judged by {judge_model}: {len(final_valuable_questions)}")


    fn_judgements = get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name, tag)
    # fn_valuable = get_valuable_file_name(question_type, difficulty, answer_model, judge_model, data_name, tag)

    async with aio_open(fn_judgements, "w") as f:
        await f.write(json.dumps(final_judgements, indent=2, ensure_ascii=False))

    # async with aio_open(fn_valuable, "w") as f:
    #     await f.write(json.dumps(final_valuable_questions, indent=2, ensure_ascii=False))

    # print(f"Finished. Saved {len(final_judgements)} judgements and {len(final_valuable_questions)} valuable questions.")
    print(f"Judgements saved to {fn_judgements}")
    # print(f"Valuable questions saved to {fn_valuable}")


In [ ]:
aaa={"short_comment": "The student identified key histological transitions and Ki-67 findings but missed specific visual phenomena such as the well-circumscribed protruding lesion with central slit-shaped cavity and signet ring cell focus details. p53 visual staining patterns were not explicitly described. Interpretations for p53 and sub-conclusions were partially correct but lacked precision. Final conclusion captures multistep progression and mucosal confinement but omits molecular distinction and independent origin of signet ring cell carcinoma.", "experiments": [{"visual_phenomenon": 0, "interpretation": 1, "sub-conclusion": 1}, {"visual_phenomenon": 0, "interpretation": 1, "sub-conclusion": 1}, {"visual_phenomenon": 1, "interpretation": -1, "sub-conclusion": 1}], "conclusion_score": 2}

In [ ]:
aaa

In [ ]:
async def single_experiment(answer_model, judge_model, question_type, difficulty, data_name, data_under_judge, image_source, tag="", rel_position=0, total_experiments=1):
    if answer_model == "fleming-38b-local":
        await answer_half_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source, tag, position=rel_position)
    elif answer_model == "fleming-8b-local":
        await answer_third_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source, tag, position=rel_position)
    else:
        await answer_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source, tag, position=rel_position)
    await judgement_main(judge_model, question_type, difficulty, answer_model, data_name, data_under_judge, image_source, tag, position=rel_position + total_experiments)

async def complete_experiment(
    answer_model_list,
    judge_model,
    question_type,
    difficulty,
    data_name,
    data_under_judge,
    image_source,
    tag="",
    rel_position=0
):
  
    tasks = []
    for i, answer_model in enumerate(answer_model_list):
        task = single_experiment(
            answer_model,
            judge_model,
            question_type,
            difficulty,
            data_name,
            data_under_judge,
            image_source,
            tag,
            rel_position=i + rel_position,
            total_experiments=len(answer_model_list)
        )
        tasks.append(task)
    await asyncio.gather(*tasks)
    

In [ ]:
# answer_model_list = ["gpt-4o-ca", "gemini-2.5-flash-ca", "claude-sonnet-4-ca", "qwen-vl-local", "lingshu-local", "lingshu-7b-local", "fleming-38b-local", "hulumed-32b-local", "hulumed-7b-local", "qwen2.5-vl-72b-local"]
answer_model_list = ["gpt-5-nano-ca"]# "gemini-3-flash-free-aihub"
judge_model = "my-deepseek"
question_type = "open" # "open" or "mc"
difficulty = "basic" # "basic" or "hard"
data_name = "final_data_v2" # file name without extension
with open(f"{data_name}.json", "r") as f:
    data_under_judge = json.load(f)
image_source = "metadata" # "folder", "metadata", or "none"

In [ ]:
# answer_model_list = ["gpt-4o-ca", "gemini-2.5-flash-ca", "claude-sonnet-4-ca", "qwen-vl-local", "lingshu-local", "lingshu-7b-local", "fleming-38b-local", "hulumed-32b-local", "hulumed-7b-local", "qwen2.5-vl-72b-local"]
answer_model_list = ["claude-opus-4-6-ca"]# "gemini-3-flash-free-aihub"
judge_model = "my-deepseek"                 
question_type = "open"  # "open" or "mc"
difficulty = "basic"  # "basic" or "hard"
data_name = "final_data_v2" 

#  1 Windows 
file_path = "./final_data_v2.json"

with open(file_path, "r", encoding="utf-8") as f:
    data_under_judge = json.load(f)

#  2 200 
data_under_judge = data_under_judge[:400]

image_source = "metadata"  # "folder", "metadata", or "none"

In [ ]:
for answer_model in answer_model_list:
    if answer_model == "fleming-38b-local":
        await answer_half_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source)
    elif answer_model == "fleming-8b-local":
        await answer_third_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source)
    else:
        await answer_main(answer_model, question_type, difficulty, data_name, data_under_judge, image_source)


In [ ]:
len(data_under_judge)

In [ ]:
await complete_experiment(
    answer_model_list,
    judge_model,
    question_type,
    difficulty,
    data_name,
    data_under_judge,
    image_source
)

In [ ]:
from math import sqrt
def calculate_final_points(judgement):
    reason_score = 0
    full_score = 0
    experiments = judgement.get("experiments", [])
    for exp in experiments:
        if int(exp.get("visual_phenomenon", 0)) == -1:
            full_score += 2
            if int(exp.get("interpretation", 0)) == 1:
                reason_score += 1
                if int(exp.get("sub-conclusion", 0)) == 1:
                    reason_score += 1
        else:
            full_score += 3
            if int(exp.get("visual_phenomenon", 0)) == 1:
                reason_score += 1
                if int(exp.get("interpretation", 0)) == 1:
                    reason_score += 1
                    if int(exp.get("sub-conclusion", 0)) == 1:
                        reason_score += 1
    # reasoning_check = judgement.get("reasoning_check", {})
    # reason_score += reasoning_check.get("content", 0)
    # reason_score += reasoning_check.get("conclusion", 0)
    # full_score += 2
    # scale to 4 points
    scale = 4 / full_score if full_score > 0 else 0
    reason_score_scaled = reason_score * scale
    conclusion_score = int(judgement.get("conclusion_score", 0))
    total_score = reason_score_scaled * conclusion_score * 4 / (reason_score_scaled + conclusion_score) if (reason_score_scaled + conclusion_score) > 0 else 0
    return {
        "total_score": total_score,
        "reason_full_score": full_score,
        "reason_score": reason_score,
        "reason_score_scaled": reason_score_scaled,
        "conclusion_score": conclusion_score,
    }

In [ ]:
# 1.  judgement 
# 
judgement_file = f"llm_judgements_{question_type}_{difficulty}_{answer_model_list[0]}_{judge_model}_{data_name}.json"

with open(judgement_file, "r", encoding="utf-8") as f:
    judgements = json.load(f)

all_normalized_scores = []

# 2. 
for entry in judgements:
    judgement = entry.get("judgement", {})
    # 
    res = calculate_final_points(judgement)

    #  [0, 4]  4 [0, 1]
    normalized_score = res["total_score"] / 4.0
    all_normalized_scores.append(normalized_score)

# 3. 
if len(all_normalized_scores) > 0:
    avg_score = sum(all_normalized_scores) / len(all_normalized_scores)
    print(f" : {len(all_normalized_scores)}")
    print(f"  (0-1): {avg_score:.4f}")
    print(f" : {avg_score * 100:.2f}")
else:
    print(" ")

In [ ]:
judgement_file = get_judgement_file_name(question_type, difficulty, answer_model_list[0], judge_model, data_name)
with open(judgement_file, "r") as f:
    judgements = json.load(f)
points = []
for entry in tqdm(judgements):
    judgement = entry.get("judgement", {})
    point = calculate_final_points(judgement)
    points.append(point)

In [ ]:
judgement_files = {
    answer_model: get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name)
    for answer_model in answer_model_list
}
all_points = {}
for answer_model, judgement_file in judgement_files.items():
    with open(judgement_file, "r") as f:
        judgements = json.load(f)
    points = []
    for entry in judgements:
        judgement = entry.get("judgement", {})
        point = calculate_final_points(judgement)
        points.append(point)
    all_points[answer_model] = points

In [ ]:
judgement_files

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# Visualize score distribution
score_bins = np.arange(0, 4.5, 0.5)
score_values = [p["total_score"] for p in points]
hist, bins = np.histogram(score_values, bins=score_bins)
plt.bar(bins[:-1], hist, width=0.5, align='edge', edgecolor='black')
plt.xlabel('Total Score')
plt.ylabel('Number of Samples')
plt.title('Score Distribution')
plt.xticks(score_bins)
plt.show()

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# Visualize score distribution for all models, each model in a subfigure
score_bins = np.arange(0, 4.5, 0.5)
num_models = len(all_points)
fig, axs = plt.subplots(1, num_models, figsize=(5 * num_models, 4), sharey=True)
if num_models == 1:
    axs = [axs]
for ax, (answer_model, points) in zip(axs, all_points.items()):
    score_values = [p["total_score"] for p in points]
    hist, bins = np.histogram(score_values, bins=score_bins)
    ax.bar(bins[:-1], hist, width=0.5, align='edge', edgecolor='black')
    ax.set_xlabel('Total Score')
    ax.set_ylabel('Number of Samples')
    ax.set_title(f'Score Distribution - {answer_model}')
    ax.set_xticks(score_bins)
plt.tight_layout()
plt.show()

In [ ]:
# statistics: 0-1, 1-2, 2-3, 3-4 total scores
score_ranges = [(0, 1), (1, 2), (2, 3), (3, 4)]
for answer_model, points in all_points.items():
    score_values = [p["total_score"] for p in points]
    total_count = len(score_values)
    sum_points = sum(score_values)
    avg_score = sum_points / total_count if total_count > 0 else 0
    print(f"Statistics for {answer_model}:")
    for r in score_ranges:
        if r[1] == 4:
            count = sum(1 for s in score_values if r[0] <= s <= r[1])
        else:
            count = sum(1 for s in score_values if r[0] <= s < r[1])
        percentage = (count / total_count) * 100 if total_count > 0 else 0
        print(f"  Score {r[0]} to {r[1]}: {count:4} samples ({percentage:.2f}%)")
    print(f"  Average Score: {avg_score:.4f}")
    # percentage
    print(f"  Average Score (100 scale): {avg_score * 25:.2f}\n")

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# Visualize score distribution for all models, each model in a subfigure
score_bins = np.arange(0, 5, 0.5)
num_models = len(all_points)
fig, axs = plt.subplots(1, num_models, figsize=(5 * num_models, 4), sharey=True)
if num_models == 1:
    axs = [axs]
for ax, (answer_model, points) in zip(axs, all_points.items()):
    score_values = [p["conclusion_score"] for p in points]
    hist, bins = np.histogram(score_values, bins=score_bins)
    ax.bar(bins[:-1], hist, width=0.5, align='edge', edgecolor='black')
    ax.set_xlabel('Conclusion Score')
    ax.set_ylabel('Number of Samples')
    ax.set_title(f'Score Distribution - {answer_model}')
    ax.set_xticks(score_bins)
plt.tight_layout()
plt.show()

In [ ]:
# statistics: 0-1, 1-2, 2-3, 3-4 total scores
score_ranges = [(0, 1), (1, 2), (2, 3), (3, 4)]
for answer_model, points in all_points.items():
    score_values = [p["conclusion_score"] for p in points]
    total_count = len(score_values)
    sum_points = sum(score_values)
    avg_score = sum_points / total_count if total_count > 0 else 0
    print(f"Statistics for {answer_model}:")
    for r in score_ranges:
        if r[1] == 4:
            count = sum(1 for s in score_values if r[0] <= s <= r[1])
        else:
            count = sum(1 for s in score_values if r[0] <= s < r[1])
        percentage = (count / total_count) * 100 if total_count > 0 else 0
        print(f"  Score {r[0]} to {r[1]}: {count:4} samples ({percentage:.2f}%)")
    print(f"  Average Score: {avg_score:.4f}")
    # percentage
    print(f"  Average Score (100 scale): {avg_score * 25:.2f}\n")

In [ ]:
# Ablation 1: Judge Conclusion only.
LLM_OPEN_JUDGE_TEMPLATE_AB1 = """
Role: Expert Biomedical Evaluator.
Task: Evaluate the quality of a [Student_Answer] strictly against the [Logic_Chain].

## Evaluation Rules
   
### Conclusion Scoring Rubric (Conclusion Accuracy 0-4)
Evaluate ONLY the quality of the Student's final conclusion compared strictly against the Logic_Chain's conclusion.
- 4 (Perfect Match): The student's conclusion identifies the exact diagnosis or result as defined in the Logic_Chain's conclusion.
- 3 (High Accuracy): Correct diagnosis, captures ALL key points from the logic chain's conclusion; but misses non-critical qualifiers.
- 2 (Partial/General): Identifies the correct general category or main disease.
- 1 (Vague/Weak): The conclusion is ambiguous or barely touches the truth.
- 0 (Mismatch): Wrong diagnosis, contradicts the Logic_Chain, or hallucinated conclusion.

### Input Data
1. Question: {question}
2. Logic_Chain: {logic_chain}
3. Student_Answer: {student_answer}


### Output Format
Return ONLY a strictly valid JSON object.

{{
  "short_comment": "<string, brief summary of judgement>",
  "conclusion_score": <integer 0-4, representing the accuracy of the final conclusion ONLY>,
}}
"""


In [ ]:
def is_valid_judgement_ab1(judgement):
    if judgement is None:
        return False
    if "conclusion_score" not in judgement:
        return False
    conclusion_score = judgement["conclusion_score"]
    if not isinstance(conclusion_score, int):
        return False
    if conclusion_score < 0 or conclusion_score > 4:
        return False
    return True

async def llm_judge_ab1_async(question, logic_chain, student_answer, images, question_type, model, client):
    if question_type == "open":
        judge_template = LLM_OPEN_JUDGE_TEMPLATE_AB1
    elif question_type == "mc":
        judge_template = LLM_MULTIPLE_CHOICE_JUDGE_TEMPLATE
    prompt = judge_template.format(
        question=question,
        logic_chain=logic_chain,
        student_answer=student_answer
    )
    messages = []
    if images:
        content = openai_pack_content(prompt, images)
    else:
        content = prompt
    messages = []
    response = await get_response_async(messages, content, model, client)
    return response

async def process_single_judgement_ab1(idx, item, answer_entry, question_type, model_name, client, judge_model, difficulty, cache_dir=""):


    if cache_dir:
        cache_file_path = os.path.join(cache_dir, f"judgement_ab2_{idx}.json")
        if os.path.exists(cache_file_path):
            async with aio_open(cache_file_path, "r") as f:
                cached_content = await f.read()
                cached_result = json.loads(cached_content)
                if cached_result:
                    is_valuable = False
                    return cached_result, is_valuable

    question = get_question(item, question_type, difficulty)
    ground_truth = get_reference_answer(item, question_type, difficulty)
    model_answer = answer_entry["model_answer"]

    input_logic_list = item.get("input_logic_chain", [])
    raw_logic_chain = input_logic_list[0] if input_logic_list and len(input_logic_list) > 0 else {}
    
  
    if model_answer is None:
        judgement_entry = {
            "question_index": idx,
            "question": question,
            "model_answer": None,
            "ground_truth": ground_truth,
            "logic_chain": raw_logic_chain,
            "judgement": {
                "conclusion_score": -1,
                "short_comment": "Model failed to give a valid answer."
            }
        }
        return judgement_entry, False


    async with sem[judge_model]:
        while True:
            try:
                judgement_result = await llm_judge_ab1_async(
                    question=question,
                    logic_chain=raw_logic_chain,
                    student_answer=model_answer,
                    images=[],
                    question_type=question_type,
                    model=model_name,
                    client=client
                )
                
                judgement = process_output(judgement_result["content"])
                
                if judgement is None:
                    print("Failed to parse judgement, retrying...")
                    await asyncio.sleep(1)
                    continue

                if not is_valid_judgement_ab1(judgement):
                    print("Invalid judgement format, retrying...")
                    await asyncio.sleep(1)
                    continue
                
                judgement_entry = {
                    "question_index": idx,
                    "question": question,
                    "model_answer": model_answer,
                    "ground_truth": ground_truth,
                    "logic_chain": raw_logic_chain,
                    "judgement": judgement
                }
                if cache_dir:
                    async with aio_open(cache_file_path, "w") as f:
                        await f.write(json.dumps(judgement_entry, indent=2, ensure_ascii=False))
                return judgement_entry, False
            except Exception as e:
                print(f"Error during judging: {e}. Retrying...")
                await asyncio.sleep(1)
                
async def judgement_main_ab1(judge_model, question_type, difficulty, answer_model, data_name, data_under_judge, image_source, tag="", position=0):

    judge_model_name = api_data["async"][judge_model]["model"]
    judge_client = api_data["async"][judge_model]["client"]


    cache_dir = get_judgement_cache_dir_name(question_type, difficulty, answer_model, judge_model, data_name, tag) + "_ab1"
    os.makedirs(cache_dir, exist_ok=True)
    
 
    answer_file = get_answer_file_name(question_type, difficulty, answer_model, data_name, tag)
    with open(answer_file, "r") as f:
        answers = json.load(f)
    tasks = []
    for idx, (item, answer_entry) in enumerate(zip(data_under_judge, answers)):
        task = process_single_judgement_ab1(idx, item, answer_entry, question_type, judge_model_name, judge_client, judge_model, difficulty, cache_dir)
        tasks.append(task)


    print(f"Starting async judgement (Ablation 1) for {len(tasks)} samples...")
    results = await tqdm_async.gather(*tasks, desc=f"Judging Samples Ablation 1 for {answer_model} with {judge_model}", position=position)


    final_judgements = []


    for i, (judgement_entry, is_valuable) in enumerate(results):
        final_judgements.append(judgement_entry)
            

    fn_judgements = get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name, tag).replace(".json", "_ab1.json")

    async with aio_open(fn_judgements, "w") as f:
        await f.write(json.dumps(final_judgements, indent=2, ensure_ascii=False))

    print(f"Judgements (Ablation 1) saved to {fn_judgements}")

In [ ]:
for answer_model in answer_model_list:
    await judgement_main_ab1(judge_model, question_type, difficulty, answer_model, data_name, data_under_judge, image_source)

In [ ]:
from math import sqrt
def calculate_final_points_ab1(judgement):
    conclusion_score = int(judgement["conclusion_score"])
    total_score = conclusion_score
    return {
        "total_score": total_score,
        "conclusion_score": conclusion_score,
    }

In [ ]:
judgement_files_ab1 = {
    answer_model: get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name).replace(".json", "_ab1.json")
    for answer_model in answer_model_list
}
all_points_ab1 = {}
for answer_model, judgement_file in judgement_files_ab1.items():
    with open(judgement_file, "r") as f:
        judgements = json.load(f)
    points = []
    for entry in judgements:
        judgement = entry.get("judgement", {})
        point = calculate_final_points_ab1(judgement)
        points.append(point)
    all_points_ab1[answer_model] = points

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# Visualize score distribution for all models, each model in a subfigure
score_bins = np.arange(0, 5, 0.5)
num_models_ab1 = len(all_points_ab1)
fig, axs = plt.subplots(1, num_models_ab1, figsize=(5 * num_models_ab1, 4), sharey=True)
if num_models_ab1 == 1:
    axs = [axs]
for ax, (answer_model, points) in zip(axs, all_points_ab1.items()):
    score_values = [p["total_score"] for p in points]
    hist, bins = np.histogram(score_values, bins=score_bins)
    ax.bar(bins[:-1], hist, width=0.5, align='edge', edgecolor='black')
    ax.set_xlabel('Total Score')
    ax.set_ylabel('Number of Samples')
    ax.set_title(f'Score Distribution - {answer_model}')
    ax.set_xticks(score_bins)
plt.tight_layout()
plt.show()

In [ ]:
# statistics: 0-1, 1-2, 2-3, 3-4 total scores
score_ranges = [(0, 1), (1, 2), (2, 3), (3, 4)]
for answer_model, points in all_points_ab1.items():
    score_values = [p["total_score"] for p in points]
    total_count = len(score_values)
    sum_points = sum(score_values)
    avg_score = sum_points / total_count if total_count > 0 else 0
    print(f"Statistics for {answer_model}:")
    for r in score_ranges:
        if r[1] == 4:
            count = sum(1 for s in score_values if r[0] <= s <= r[1])
        else:
            count = sum(1 for s in score_values if r[0] <= s < r[1])
        percentage = (count / total_count) * 100 if total_count > 0 else 0
        print(f"  Score {r[0]} to {r[1]}: {count:4} samples ({percentage:.2f}%)")
    print(f"  Average Score: {avg_score:.4f}")
    # percentage
    print(f"  Average Score (100 scale): {avg_score * 25:.2f}\n")

In [ ]:
# Ablation 2: Use Reference_Answer instead of Logic_Chain for judgement. Judge Reasoning and Conclusion.
LLM_OPEN_JUDGE_TEMPLATE_AB2 = """
Role: Expert Biomedical Evaluator.
Task: Evaluate the quality of a [Student_Answer] strictly against the [Reference_Answer].

## Evaluation Rules

### Reasoning Scoring Rubric (Reasoning Accuracy 0-4)
Evaluate the overall reasoning quality of the Student's answer based on the evidence and logic presented in the Reference_Answer.
- 4 (Perfect Reasoning): The student's reasoning perfectly aligns with the reference, capturing all critical evidence and logical steps.
- 3 (Strong Reasoning): The student's reasoning is mostly accurate, with minor omissions or slight misinterpretations of non-critical evidence.
- 2 (Moderate Reasoning): The student's reasoning captures some key points but misses significant evidence or contains notable logical gaps.
- 1 (Weak Reasoning): The student's reasoning is largely flawed, missing most critical evidence or containing major logical errors.
- 0 (No Reasoning): The student's reasoning is entirely incorrect or absent, showing no alignment with the reference.
   
### Conclusion Scoring Rubric (Conclusion Accuracy 0-4)
Evaluate ONLY the quality of the Student's final conclusion compared strictly against the Reference_Answer's conclusion.
- 4 (Perfect Match): The student's conclusion identifies the exact diagnosis or result as defined in the Reference_Answer's conclusion.
- 3 (High Accuracy): Correct diagnosis, captures ALL key points from the reference; but misses non-critical qualifiers.
- 2 (Partial/General): Identifies the correct general category or main disease.
- 1 (Vague/Weak): The conclusion is ambiguous or barely touches the truth.
- 0 (Mismatch): Wrong diagnosis, contradicts the Reference_Answer, or hallucinated conclusion.

### Input Data
1. Question: {question}
2. Reference_Answer: {reference_answer}
3. Student_Answer: {student_answer}


### Output Format
Return ONLY a strictly valid JSON object.

{{
  "short_comment": "<string, brief summary of judgement>",
  "reasoning_score": <integer 0-4, representing the overall reasoning accuracy>,
  "conclusion_score": <integer 0-4, representing the accuracy of the final conclusion ONLY>,
}}
"""


In [ ]:
def is_valid_judgement_ab2(judgement):
    if judgement is None:
        return False
    if "reasoning_score" not in judgement:
        return False
    if "conclusion_score" not in judgement:
        return False
    reasoning_score = judgement["reasoning_score"]
    conclusion_score = judgement["conclusion_score"]
    if not isinstance(reasoning_score, int):
        return False
    if not isinstance(conclusion_score, int):
        return False
    if reasoning_score < 0 or reasoning_score > 4:
        return False
    if conclusion_score < 0 or conclusion_score > 4:
        return False
    return True

async def llm_judge_ab2_async(question, reference_answer, student_answer, images, question_type, model, client):
    if question_type == "open":
        judge_template = LLM_OPEN_JUDGE_TEMPLATE_AB2
    elif question_type == "mc":
        judge_template = LLM_MULTIPLE_CHOICE_JUDGE_TEMPLATE
    prompt = judge_template.format(
        question=question,
        reference_answer=reference_answer,
        student_answer=student_answer
    )
    messages = []
    if images:
        content = openai_pack_content(prompt, images)
    else:
        content = prompt
    messages = []
    response = await get_response_async(messages, content, model, client)
    return response

async def process_single_judgement_ab2(idx, item, answer_entry, question_type, model_name, client, judge_model, difficulty, cache_dir=""):
  

    if cache_dir:
        cache_file_path = os.path.join(cache_dir, f"judgement_ab2_{idx}.json")
        if os.path.exists(cache_file_path):
            async with aio_open(cache_file_path, "r") as f:
                cached_content = await f.read()
                cached_result = json.loads(cached_content)
                if cached_result:
                    is_valuable = False
                    return cached_result, is_valuable

    question = get_question(item, question_type, difficulty)
    ground_truth = get_reference_answer(item, question_type, difficulty)
    model_answer = answer_entry["model_answer"]

    input_logic_list = item.get("input_logic_chain", [])
    raw_logic_chain = input_logic_list[0] if input_logic_list and len(input_logic_list) > 0 else {}
    

    if model_answer is None:
        judgement_entry = {
            "question_index": idx,
            "question": question,
            "model_answer": None,
            "ground_truth": ground_truth,
            "logic_chain": raw_logic_chain,
            "judgement": {
                "reasoning_score": -1,
                "conclusion_score": -1,
                "short_comment": "Model failed to give a valid answer."
            }
        }
        return judgement_entry, False


    async with sem[judge_model]:
        while True:
            try:
                judgement_result = await llm_judge_ab2_async(
                    question=question,
                    reference_answer=ground_truth,
                    student_answer=model_answer,
                    images=[],
                    question_type=question_type,
                    model=model_name,
                    client=client
                )
                
                judgement = process_output(judgement_result["content"])
                
                if judgement is None:
                    print("Failed to parse judgement, retrying...")
                    await asyncio.sleep(1)
                    continue

                if not is_valid_judgement_ab2(judgement):
                    print("Invalid judgement format, retrying...")
                    await asyncio.sleep(1)
                    continue
                
                judgement_entry = {
                    "question_index": idx,
                    "question": question,
                    "model_answer": model_answer,
                    "ground_truth": ground_truth,
                    "logic_chain": raw_logic_chain,
                    "judgement": judgement
                }
                if cache_dir:
                    async with aio_open(cache_file_path, "w") as f:
                        await f.write(json.dumps(judgement_entry, indent=2, ensure_ascii=False))
                return judgement_entry, False
            except Exception as e:
                print(f"Error during judging: {e}. Retrying...")
                await asyncio.sleep(1)
                
async def judgement_main_ab2(judge_model, question_type, difficulty, answer_model, data_name, data_under_judge, image_source, tag="", position=0):

    judge_model_name = api_data["async"][judge_model]["model"]
    judge_client = api_data["async"][judge_model]["client"]


    cache_dir = get_judgement_cache_dir_name(question_type, difficulty, answer_model, judge_model, data_name, tag) + "_ab2"
    os.makedirs(cache_dir, exist_ok=True)
    
   
    answer_file = get_answer_file_name(question_type, difficulty, answer_model, data_name, tag)
    with open(answer_file, "r") as f:
        answers = json.load(f)
    tasks = []
    for idx, (item, answer_entry) in enumerate(zip(data_under_judge, answers)):
        task = process_single_judgement_ab2(idx, item, answer_entry, question_type, judge_model_name, judge_client, judge_model, difficulty, cache_dir)
        tasks.append(task)

   
    print(f"Starting async judgement (Ablation 2) for {len(tasks)} samples...")
    results = await tqdm_async.gather(*tasks, desc=f"Judging Samples Ablation 2 for {answer_model} with {judge_model}", position=position)


    final_judgements = []

    for i, (judgement_entry, is_valuable) in enumerate(results):
        final_judgements.append(judgement_entry)
            
    fn_judgements = get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name, tag).replace(".json", "_ab2.json")

    async with aio_open(fn_judgements, "w") as f:
        await f.write(json.dumps(final_judgements, indent=2, ensure_ascii=False))

    print(f"Judgements (Ablation 2) saved to {fn_judgements}")

In [ ]:
for answer_model in answer_model_list:
    await judgement_main_ab2(judge_model, question_type, difficulty, answer_model, data_name, data_under_judge, image_source)

In [ ]:
from math import sqrt
def calculate_final_points_ab2(judgement):
    reason_score = int(judgement["reasoning_score"])
    conclusion_score = int(judgement["conclusion_score"])
    total_score = reason_score * conclusion_score / 4
    return {
        "total_score": total_score,
        "reason_score": reason_score,
        "conclusion_score": conclusion_score,
    }

In [ ]:
from math import sqrt
def calculate_final_points_ab2_add(judgement):
    reason_score = int(judgement["reasoning_score"])
    conclusion_score = int(judgement["conclusion_score"])
    total_score = reason_score * 0.5 + conclusion_score * 0.5
    return {
        "total_score": total_score,
        "reason_score": reason_score,
        "conclusion_score": conclusion_score,
    }

In [ ]:
judgement_files_ab2 = {
    answer_model: get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name).replace(".json", "_ab2.json")
    for answer_model in answer_model_list
}
all_points_ab2 = {}
for answer_model, judgement_file in judgement_files_ab2.items():
    with open(judgement_file, "r") as f:
        judgements = json.load(f)
    points = []
    for entry in judgements:
        judgement = entry.get("judgement", {})
        point = calculate_final_points_ab2(judgement)
        points.append(point)
    all_points_ab2[answer_model] = points

In [ ]:
all_points_ab2_add = {}
for answer_model, judgement_file in judgement_files_ab2.items():
    with open(judgement_file, "r") as f:
        judgements = json.load(f)
    points = []
    for entry in judgements:
        judgement = entry.get("judgement", {})
        point = calculate_final_points_ab2_add(judgement)
        points.append(point)
    all_points_ab2_add[answer_model] = points

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# Visualize score distribution for all models, each model in a subfigure
score_bins = np.arange(0, 5, 0.5)
num_models_ab2 = len(all_points_ab2)
fig, axs = plt.subplots(1, num_models_ab2, figsize=(5 * num_models_ab2, 4), sharey=True)
if num_models_ab2 == 1:
    axs = [axs]
for ax, (answer_model, points) in zip(axs, all_points_ab2.items()):
    score_values = [p["total_score"] for p in points]
    hist, bins = np.histogram(score_values, bins=score_bins)
    ax.bar(bins[:-1], hist, width=0.5, align='edge', edgecolor='black')
    ax.set_xlabel('Total Score')
    ax.set_ylabel('Number of Samples')
    ax.set_title(f'Score Distribution - {answer_model}')
    ax.set_xticks(score_bins)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# Visualize score distribution for all models, each model in a subfigure
score_bins = np.arange(0, 5, 0.5)
num_models_ab2_add = len(all_points_ab2_add)
fig, axs = plt.subplots(1, num_models_ab2_add, figsize=(5 * num_models_ab2_add, 4), sharey=True)
if num_models_ab2_add == 1:
    axs = [axs]
for ax, (answer_model, points) in zip(axs, all_points_ab2_add.items()):
    score_values = [p["total_score"] for p in points]
    hist, bins = np.histogram(score_values, bins=score_bins)
    ax.bar(bins[:-1], hist, width=0.5, align='edge', edgecolor='black')
    ax.set_xlabel('Total Score')
    ax.set_ylabel('Number of Samples')
    ax.set_title(f'Score Distribution - {answer_model}')
    ax.set_xticks(score_bins)
plt.tight_layout()
plt.show()

In [ ]:
# statistics: 0-1, 1-2, 2-3, 3-4 total scores
score_ranges = [(0, 1), (1, 2), (2, 3), (3, 4)]
for answer_model, points in all_points_ab2.items():
    score_values = [p["total_score"] for p in points]
    total_count = len(score_values)
    sum_points = sum(score_values)
    avg_score = sum_points / total_count if total_count > 0 else 0
    print(f"Statistics for {answer_model}:")
    for r in score_ranges:
        if r[1] == 4:
            count = sum(1 for s in score_values if r[0] <= s <= r[1])
        else:
            count = sum(1 for s in score_values if r[0] <= s < r[1])
        percentage = (count / total_count) * 100 if total_count > 0 else 0
        print(f"  Score {r[0]} to {r[1]}: {count:4} samples ({percentage:.2f}%)")
    print(f"  Average Score: {avg_score:.4f}")
    # percentage
    print(f"  Average Score (100 scale): {avg_score * 25:.2f}\n")

In [ ]:
# statistics: 0-1, 1-2, 2-3, 3-4 conclusion scores
score_ranges = [(0, 1), (1, 2), (2, 3), (3, 4)]
for answer_model, points in all_points_ab2.items():
    score_values = [p["conclusion_score"] for p in points]
    total_count = len(score_values)
    sum_points = sum(score_values)
    avg_score = sum_points / total_count if total_count > 0 else 0
    print(f"Statistics for {answer_model}:")
    for r in score_ranges:
        if r[1] == 4:
            count = sum(1 for s in score_values if r[0] <= s <= r[1])
        else:
            count = sum(1 for s in score_values if r[0] <= s < r[1])
        percentage = (count / total_count) * 100 if total_count > 0 else 0
        print(f"  Score {r[0]} to {r[1]}: {count:4} samples ({percentage:.2f}%)")
    print(f"  Average Score: {avg_score:.4f}")
    # percentage
    print(f"  Average Score (100 scale): {avg_score * 25:.2f}\n")

In [ ]:
def get_filtered_indices(file_name_list, threshold=2):
    filtered_sets = []
    for file_name in file_name_list:
        with open(file_name, "r") as f:
            judgement_json = json.load(f)
        filtered_indices = set(entry["question_index"] for entry in judgement_json if calculate_final_points(entry["judgement"])["conclusion_score"] < threshold)
        filtered_sets.append(filtered_indices)

    common_filtered_indices = set.intersection(*filtered_sets)
    return common_filtered_indices

async def generate_filtered_question_indices(
    answer_model_list,
    judge_model,
    question_type,
    difficulty,
    data_name,
    data_under_judge,
    image_source,
    tag="no_images",
    threshold=2
):


    judgement_file_names = [
        get_judgement_file_name(question_type, difficulty, answer_model, judge_model, data_name, tag)
        for answer_model in answer_model_list
    ]


    common_filtered_indices = get_filtered_indices(judgement_file_names, threshold)
    

    return common_filtered_indices

In [ ]:
def filter_entries(target_data, filtered_indices):
    for entry in target_data:
        if "question_index" not in entry:
            entry["question_index"] = target_data.index(entry)
    return [entry for entry in target_data if entry["question_index"] in filtered_indices]

def filter_file(input_file, output_file, filtered_indices):
    with open(input_file, "r") as f:
        data = json.load(f)
    for idx, entry in enumerate(data):
        if "question_index" not in entry:
            entry["question_index"] = idx
    filtered_data = filter_entries(data, filtered_indices)
    with open(output_file, "w") as f:
        json.dump(filtered_data, f, indent=2, ensure_ascii=False)
    # print(f"Filtered data saved to {output_file}, total {len(filtered_data)} entries.")

def judgement_statistics(judge_json):
    #sum_score = 0
    #for score in range(-1, 5):
    #    count = sum(1 for entry in judge_json if entry["judgement"]["correctness"] == score)
    #    print(f"Score {score}: {count} samples")
    #    sum_score += score * count
    #total_samples = len(judge_json)
    #average_score = sum_score / total_samples if total_samples > 0 else 0
    #print(f"Average Correctness Score: {average_score:.2f} over {total_samples} samples")
    pass

In [ ]:
async def filter_main(
    experiment_params,
    filter_groups
):
    results = []
    for filter_group in filter_groups:
        task = await generate_filtered_question_indices(
            filter_group["answer_model_list"],
            filter_group["judge_model"],
            experiment_params["question_type"],
            experiment_params["difficulty"],
            experiment_params["data_name"],
            experiment_params["data_under_judge"],
            filter_group["image_source"],
            filter_group["tag"],
            filter_group["threshold"]
        )
        results.append(task)

    # 
    for i, filter_group in enumerate(filter_groups):
        group_filtered_indices = results[i]
        print(f"Filter Group {i+1} {filter_group.get('tag', '')}:")
        print(f"Total common filtered indices: {len(group_filtered_indices)}")
        group_valuable_questions = filter_entries(
            experiment_params["data_under_judge"],
            group_filtered_indices
        )
        group_valuable_file = f"filtered_valuable_questions_group{i+1}_{experiment_params['question_type']}_{experiment_params['difficulty']}_{experiment_params['data_name']}{('_' + filter_group.get('tag', '')) if filter_group.get('tag', '') else ''}.json"
        with open(group_valuable_file, "w") as f:
            json.dump(group_valuable_questions, f, indent=2, ensure_ascii=False)
        print(f"Filtered valuable questions for group {i+1} saved to {group_valuable_file}, total {len(group_valuable_questions)} entries.")
        # filter judgement files
        # for answer_model in experiment_params["answer_model_list"]:
        #     judgement_file = get_judgement_file_name(
        #         experiment_params["question_type"],
        #         experiment_params["difficulty"],
        #         answer_model,
        #         experiment_params["judge_model"],
        #         experiment_params["data_name"],
        #         experiment_params.get("tag", "")
        #     )
        #     judgement_prefix = judgement_file[:-5]  # remove .json
        #     output_file = f"filtered_{judgement_prefix}{'_' if filter_group.get('tag', '') else ''}{filter_group.get('tag', '')}.json"
        #     filter_file(judgement_file, output_file, group_filtered_indices)
        #     with open(output_file, "r") as f:
        #         judge_json = json.load(f)
        #     print(f"Statistics for {output_file} group {filter_group.get('tag', '')}:")
        #     judgement_statistics(judge_json)
        print("==============================")
        

    common_filtered_indices = set.intersection(*results)
    
    print("Overall Common Filtered Indices Across All Groups:")
    print(f"Total common filtered indices across all groups: {len(common_filtered_indices)}")

    # filter valuable questions
    valuable_questions = filter_entries(
        experiment_params["data_under_judge"],
        common_filtered_indices
    )
    valuable_file = f"filtered_valuable_questions_{experiment_params['question_type']}_{experiment_params['difficulty']}_{experiment_params['data_name']}{('_' + experiment_params.get('tag', '')) if experiment_params.get('tag', '') else ''}.json"
    with open(valuable_file, "w") as f:
        json.dump(valuable_questions, f, indent=2, ensure_ascii=False)
    print(f"Filtered valuable questions saved to {valuable_file}, total {len(valuable_questions)} entries.")
    # filter judgement files
    # for answer_model in experiment_params["answer_model_list"]:
    #     judgement_file = get_judgement_file_name(
    #         experiment_params["question_type"],
    #         experiment_params["difficulty"],
    #         answer_model,
    #         experiment_params["judge_model"],
    #         experiment_params["data_name"],
    #         experiment_params.get("tag", "")
    #     )
    #     judgement_prefix = judgement_file[:-5]  # remove .json
    #     output_file = f"filtered_{judgement_prefix}_all_groups.json"
    #     filter_file(judgement_file, output_file, common_filtered_indices)
    #     with open(output_file, "r") as f:
    #         judge_json = json.load(f)
    #     print(f"Statistics for {output_file}:")
    #     judgement_statistics(judge_json)
        

In [ ]:
experiment_params = {
    "answer_model_list": answer_model_list,
    "judge_model": judge_model,
    "question_type": question_type,
    "difficulty": difficulty,
    "data_name": data_name,
    "data_under_judge": data_under_judge,
    "tag": ""
}

filter_groups = {
    "no_images": {
        "answer_model_list": ["deepseek-v3.2-ca"],
        "judge_model": "deepseek-v3.2-ca",
        "image_source": "none",
        "tag": "no_images",
        "threshold": 4
    },
    # "weak_model": {
    #     "answer_model_list": ["qwen3-vl-32b-local"],
    #     "judge_model": "deepseek-v3.2-ca",
    #     "image_source": "metadata",
    #     "tag": "weak_model",
    #     "threshold": 4
    # }
}

In [ ]:
difficulty

In [ ]:
get_judgement_file_name(
                experiment_params["question_type"],
                experiment_params["difficulty"],
                answer_model_list[0],
                experiment_params["judge_model"],
                experiment_params["data_name"],
                experiment_params.get("tag", "")
            )

In [ ]:
tasks = []
for i, key in enumerate(filter_groups):
    filter_group = filter_groups[key]
    task = complete_experiment(
        filter_group["answer_model_list"],
        filter_group["judge_model"],
        experiment_params["question_type"],
        experiment_params["difficulty"],
        experiment_params["data_name"],
        experiment_params["data_under_judge"],
        filter_group["image_source"],
        filter_group["tag"],
        rel_position=i*len(filter_group["answer_model_list"])
    )
    tasks.append(task)
await asyncio.gather(*tasks)

In [ ]:
await filter_main(
    experiment_params,
    list(filter_groups.values())
)

In [ ]:
with open("filtered_llm_judgements_open_basic_lingshu_deepseek-v3.2-ca_step7_logic_based_qa_output_processed_qc3_passed_2.json", "r") as f:
    judge_json = json.load(f)
for entry in judge_json:
    entry["original_sample_index"] = data_under_judge[entry["question_index"]]["original_sample_index"]
with open("filtered_llm_judgements_open_basic_lingshu_deepseek-v3.2-ca_step7_logic_based_qa_output_processed_qc3_passed_2.json", "w") as f:
    json.dump(judge_json, f, indent=2, ensure_ascii=False)

In [ ]:
import json

In [ ]:
open_question_file = "step7_logic_based_qa_output_2_processed_qc3_passed.json"
with open(open_question_file, "r") as f:
    open_questions = json.load(f)

In [ ]:
len(open_questions)

In [ ]:
open_questions[0].keys()

In [ ]:
open_questions[0]

In [ ]:
open_questions[0]

In [ ]:
all_metadata[str(open_questions[100]["original_sample_index"])]["image_info"]

In [ ]:
def open_question_item_to_md(entry):
    md_content = f"### Question Index: {entry.get('question_index', 'N/A')}\n\n"
    md_content += f"**Original Sample Index:** {entry.get('original_sample_index', 'N/A')}\n\n"
    md_content += f"**Context:**\n\n{entry.get('input_context', 'N/A')}\n\n"
    md_content += f"**Observation:**\n\n{entry.get('input_observation', 'N/A')}\n\n"
    md_content += f"**Images:**\n\n"
    image_info = all_metadata[str(entry.get("original_sample_index", -1))]["image_info"]
    for img in image_info:
        md_content += f"Image {img['index']}\n"
        md_content += f"<img src=\"data:image/jpeg;base64,{img['image_base64']}\" alt=\"Image {img['index']}\" style=\"max-width: 600px;\" />\n\n"
        md_content += f"**Caption:** {img['caption']}\n\n"
    logic_chain = entry.get('input_logic_chain', [])
    if logic_chain:
        logic_chain = logic_chain[0]
        research_context = logic_chain.get('research_context', '')
        md_content += f"**Research Context:**\n\n{research_context}\n\n"
        experiments = logic_chain.get('experiments', [])
        for i, exp in enumerate(experiments):
            md_content += f"**Experiment {i+1}:**\n\n"
            md_content += f"- **Setting:** {exp.get('experimental_setting', '')}\n"
            md_content += f"- **Goal:** {exp.get('experiment_goal', '')}\n"
            md_content += f"- **Visual Phenomenon:** {exp.get('visual_phenomenon', '')}\n"
            md_content += f"- **Interpretation:** {exp.get('interpretation', '')}\n\n"
            md_content += f"- **Sub Conclusion:** {exp.get('sub_conclusion', '')}\n\n"
        reasoning = logic_chain.get('reasoning', '')
        md_content += f"**Overall Reasoning:**\n\n{reasoning.get('content', '')}\n\n"
        md_content += f"**Final Conclusion:**\n\n{reasoning.get('conclusion', '')}\n\n"
    md_content += f"**Question:**\n\n{entry['basic_qa']['question']}\n\n"
    md_content += f"**Answer:**\n\n{entry['basic_qa']['answer']}\n\n"
    return md_content

In [ ]:
all_md_content = ""
for idx, entry in enumerate(open_questions):
    entry["question_index"] = idx
    all_md_content += open_question_item_to_md(entry)
    all_md_content += "\n---\n\n"

In [ ]:
with open("164_open_questions.md", "w") as f:
    f.write(all_md_content)